# Metropolis escape from deep cubic basins: numerical calculations and figures

This notebook computes the mean escape time $H_D(N)$ of the Metropolis walk and the shell-quotient mean $Q_D(N)$.
Time is measured in attempted moves, including rejected proposals.

The true-chain calculation uses exact sign/permutation-orbit lumping and the regenerative identity
$H_D(N)=(1+p\,g_{ex}(e_1))/(p\,h_{ex}(e_1))$.
The float64 sparse LU solver checks componentwise backward errors against $10^{-9}$ and, by default,
agreement between two LU orderings against $10^{-8}$. A failed check raises an exception.
All plotted $H_D$ values use this solver; $Q_D$ uses the closed-form sum in Theorem 1.
Selected true-chain results are independently checked with a block-Thomas calculation at 60 and 90 digits.
Set `RUN_MPMATH_AUDIT = False` to skip these optional checks.

## Outputs

Figures 4 and 5 reproduce the paper's numerical figures, and ten checks confirm the numerical values quoted in Section VI.
The paper's Figures 1–3 illustrate the geometry and network arguments; they are separate from the supplementary plots below.
Supplementary Figures S1–S3 are additional repository plots, not figures in the paper or a claim of a separately published supplement.

| Figure | Content | Parameter grid |
|---|---|---|
| S1 | $H_D$ and $Q_D$ versus depth; signed discrepancy | $D=1,2,3$; $p=0.2,0.55,0.9$; $N=2,3,5,8,11,14,17,20,23,25$ |
| S2 | Dimensional dependence of $H_D$ and $Q_D$ | $N=20$, 50 values of $p\in[0.15,0.90]$; $p=0.55$, $N=5,\ldots,25$ |
| S3 | $H_D$ and $Q_D$ versus acceptance probability; signed discrepancy | $D=1,2,3$; $N=5,15,25$; $p=0.15,0.20,\ldots,0.90$ |
| 4 | True escape times versus depth and dimensional speedup versus acceptance probability | Same grid as S2 |
| 5 | True and quotient times versus depth; signed discrepancy | Same grid as S1 |

S3 starts at $p=0.15$: the float64 solver's residual check fails at $N=25$, $p=0.10$ for $D=2,3$.
The shell-quotient formula equals the true-chain mean for $D=1$; for $D>1$ it describes a different chain.
Numerical comparisons between $Q_D$ and $H_D$ are observations on the configured grid, not a general inequality.

Run all cells in order. Results are saved under `metropolis_results/`: CSV data, PDF/SVG/PNG figures,
paper figure captions and JSON manifests. Supplementary plots are in `figures/supplementary/`.
The final cell creates `metropolis_results.zip` and downloads it automatically in Google Colab.


### 1. Configuration

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec('mpmath') is None:          # Colab already has it
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'mpmath'])

import itertools, json, time, zipfile
from datetime import datetime, timezone
from pathlib import Path
import numpy as np, pandas as pd, scipy, matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.sparse import coo_matrix
from scipy.sparse.linalg import splu, spsolve
import mpmath as mp

OUTPUT_DIR = Path('metropolis_results').resolve()

# ---- Supplementary Figure S1 --------------------------------------------
SUPP1_D = (1, 2, 3)
SUPP1_P = (0.20, 0.55, 0.90)
SUPP1_N = (2, 3, 5, 8, 11, 14, 17, 20, 23, 25)
# SUPP1_N = tuple(range(2, 26))                    # denser alternative

# ---- Supplementary Figure S2 --------------------------------------------
SUPP2_D = (1, 2, 3)
SUPP2A_N = 20
SUPP2A_P = tuple(np.round(np.linspace(0.15, 0.90, 50), 6))
SUPP2B_P = 0.55
SUPP2B_N = tuple(range(5, 26))

# ---- Supplementary Figure S3 --------------------------------------------
SUPP3_D = (1, 2, 3)
SUPP3_N = (5, 15, 25)
SUPP3_P = (0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55,
          0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90)
# p=0.10 is excluded: the residual check fails for D=2,3 at N=25.
EXTEND_SUPP3_TO_P1 = False   # True adds p = 0.95, 0.975, 1.0

# ---- Numerical validation -----------------------------------------------
CHECK_ALTERNATE_LU_ORDERING = True  # second LU ordering must agree to 1e-8
RUN_MPMATH_AUDIT = True             # False skips the independent precision checks
AUDIT_DPS = (60, 90)                # each audit is repeated at both precisions
AUDIT_CASES = ((2, 23, 0.20),       # reference case
               (3, 8, 0.20),        # reference case
               (2, 25, 0.15),       # deep D=2 case (H ~ 2.6e19, Supplementary Figure S3)
               (3, 10, 0.15))       # low-p D=3 case
RUN_HARDEST_D3_AUDIT = False        # expensive dense-block check at (3,25,0.15)

if EXTEND_SUPP3_TO_P1:
    SUPP3_P = SUPP3_P + (0.95, 0.975, 1.0)
for sub in ('data', 'figures', 'figures/supplementary'):
    (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)
plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 9, 'axes.titlesize': 10,
    'axes.labelsize': 9, 'legend.fontsize': 8, 'xtick.labelsize': 8,
    'ytick.labelsize': 8, 'pdf.fonttype': 42, 'ps.fonttype': 42,
    'figure.dpi': 130, 'savefig.bbox': 'tight'})
print('Python', sys.version.split()[0], '| NumPy', np.__version__,
      '| SciPy', scipy.__version__, '| mpmath', mp.__version__)


### 2. Shell-quotient formula $Q_D(N)$
Implements Theorem 1, Eq. (17): the exact mean escape time for the shell-quotient chain, equal to the true-chain mean for $D=1$.


In [ ]:
def shell_size(m: int, D: int) -> int:
    """Number of lattice points with ||x||_inf = m (m >= 1)."""
    if m < 1 or D < 1:
        raise ValueError('Require m >= 1 and D >= 1')
    return (2*m + 1)**D - (2*m - 1)**D


def quotient_hitting_time(N: int, p: float, D: int) -> float:
    """Exact mean exit time of the shell-shorted / shell-averaged chain.

    For D>1 this is not the exact true
    Metropolis hitting time, since the shell projection is not Markov.

    The form below factors alpha^(ell+1), avoiding a double sum of huge
    terms and making the role of F_ell = 1 + sum_m S_D(m) p^m explicit.
    """
    if N < 1 or D < 1 or not (0 < p <= 1):
        raise ValueError('Require N >= 1, D >= 1 and 0 < p <= 1')
    F = 1.0
    total = 0.0
    alpha = 1.0 / p
    power = 1.0
    for ell in range(N):
        if ell > 0:
            F += shell_size(ell, D) * p**ell
        power *= alpha
        total += power * F / (2*ell + 1)**(D-1)
    return total


### 3. True-chain solver
Exact sign/permutation-orbit lumping with the regenerative identity
$H_D(N)=(1+p\,g_{ex}(e_1))/(p\,h_{ex}(e_1))$.


In [ ]:
def orbit_states(N: int, D: int):
    """All transient sign/permutation orbits, excluding the origin.

    Canonical representative is sorted absolute coordinates in ascending
    order, e.g. (0,1,3). This is an exact lumping by symmetries.
    The state count (including the origin) is binomial(N+D-1,D).
    """
    return [s for s in itertools.combinations_with_replacement(range(N), D)
            if s[-1] != 0]


def make_excursion_system(N: int, p: float, D: int):
    """Build (I-P_int)*h=b and (I-P_int)*g=1 on non-origin orbits.

    Origin and shell N are absorbing boundaries for each excursion.
    h(x)=P_x(hit shell N before returning to origin), and
    g(x)=E_x[min(tau_origin,tau_shell_N)].

    When an uphill proposal is rejected, the walk stays at x. This
    self-loop must be represented, including on moves toward shell N.
    """
    if not (N >= 2 and D in (1,2,3) and 0 < p <= 1):
        raise ValueError('Require N>=2, D=1,2,3 and 0<p<=1')
    states = orbit_states(N, D)
    loc = {s: i for i, s in enumerate(states)}
    n = len(states)
    rows, cols, weights = [], [], []
    to_outer = np.zeros(n, dtype=float)
    to_origin = np.zeros(n, dtype=float)
    step = 1.0 / (2*D)

    for i, state in enumerate(states):
        radius = state[-1]
        rows.append(i); cols.append(i); weights.append(1.0)
        for coordinate in range(D):
            for direction in (-1, +1):
                neighbor = list(state)
                neighbor[coordinate] += direction
                new_radius = max(abs(z) for z in neighbor)
                uphill = new_radius > radius
                accept = p if uphill else 1.0
                # The rejected move is a real self-loop, not absorption.
                if uphill:
                    rows.append(i); cols.append(i)
                    weights.append(-(1.0-p)*step)
                transition = accept * step
                if new_radius == N:
                    to_outer[i] += transition
                elif new_radius == 0:
                    to_origin[i] += transition
                else:
                    canonical = tuple(sorted(abs(z) for z in neighbor))
                    rows.append(i); cols.append(loc[canonical])
                    weights.append(-transition)

    A = coo_matrix((weights, (rows, cols)), shape=(n, n)).tocsc()
    # Independent stochasticity check: A*1 = probability of leaving
    # the excursion state set in one step.
    conservation_error = np.max(np.abs(
        A.dot(np.ones(n)) - (to_origin + to_outer)))
    if conservation_error > 1e-12:
        raise AssertionError(f'Transition conservation failed: {conservation_error}')
    return A, to_outer, to_origin, loc, conservation_error


def componentwise_backward_error(A, x, b) -> float:
    """Componentwise backward error; does not bound forward error."""
    residual = np.abs(A.dot(x) - b)
    denominator = abs(A).dot(np.abs(x)) + np.abs(b)
    return float(np.max(residual / np.maximum(denominator, 1e-300)))


def true_hitting_time(N: int, p: float, D: int, check_alt=True) -> dict:
    """Mean true-chain exit time from the origin, in attempted steps.

    This regenerative computation avoids the nearly singular direct
    mean-hitting-time system (I-P)T=1 at huge escape times.
    It uses the exactly lumped orbit chain, not shell averaging.
    """
    if N == 1:
        return dict(H_true=1/p, orbit_count=0, h_neighbor=1.,
                    g_neighbor=0., error_h=0., error_g=0.,
                    alternative_relative_difference=0.)
    A, b, to_origin, index, conservation_error = make_excursion_system(N,p,D)
    ones = np.ones(A.shape[0])
    lu = splu(A, permc_spec='COLAMD')
    solution = lu.solve(np.column_stack([b, ones]))
    h, g = solution[:, 0], solution[:, 1]
    first_neighbor = (0,)*(D-1) + (1,)
    j = index[first_neighbor]
    if h[j] <= 0 or g[j] < 0 or np.min(h) < -1e-10:
        raise ArithmeticError('Nonphysical excursion solution; inspect conditioning')
    H = (1 + p*g[j])/(p*h[j])
    if not np.isfinite(H) or H <= 0:
        raise ArithmeticError('Invalid true hitting time')
    err_h = componentwise_backward_error(A,h,b)
    err_g = componentwise_backward_error(A,g,ones)
    if max(err_h,err_g) > 1e-9:
        raise ArithmeticError(f'Unacceptable sparse-solve residual: {err_h}, {err_g}')
    alt_diff = np.nan
    if check_alt:
        # M-matrix has positive pivots; avoid unhelpful pivoting and
        # change the permutation ordering for an independent factorization.
        alt = splu(A, permc_spec='MMD_ATA', diag_pivot_thresh=0.0).solve(
            np.column_stack([b,ones]))
        H_alt = (1 + p*alt[j,1])/(p*alt[j,0])
        alt_diff = abs(H_alt/H - 1)
        if alt_diff > 1e-8:
            raise ArithmeticError(f'Independent LU orderings disagree by {alt_diff}')
    return dict(H_true=float(H), orbit_count=A.shape[0],
                h_neighbor=float(h[j]), g_neighbor=float(g[j]),
                error_h=err_h, error_g=err_g,
                alternative_relative_difference=float(alt_diff))


### 4. High-precision block-Thomas reference
An independent direct solve of the orbit-chain hitting-time equations, used to check selected float64 results at 60 and 90 digits.


In [ ]:
def high_precision_block_reference(N: int, p: float, D: int, dps: int=60) -> float:
    """Independent high-precision orbit-block Thomas hitting-time solve.

    Solves (I-P)T=1 directly using mpmath arithmetic on each shell.
    Dense blocks make large D=3 cases expensive; use selected cases.
    """
    if D not in (2,3):
        raise ValueError('High-precision block audit supports D=2 or 3')
    with mp.workdps(dps):
        pv = mp.mpf(str(p)); weight = mp.mpf(1)/(2*D)
        shells = [list(itertools.combinations_with_replacement(range(r+1), D-1))
                  for r in range(N)]
        shells = [ [tuple(s)+(r,) for s in group] for r,group in enumerate(shells)]
        # Tuple is ascending; e.g., D=3 shell 3 gives (c,b,3), c<=b<=3.
        where = [{s:i for i,s in enumerate(group)} for group in shells]
        alpha, beta = [None]*N,[None]*N
        for r in range(N):
            nr = len(shells[r]); nxt = len(shells[r+1]) if r+1<N else 0
            prv = len(shells[r-1]) if r>0 else 0
            B=mp.eye(nr)
            U=mp.matrix(nr,nxt) if nxt else None
            L=mp.matrix(nr,prv) if prv else None
            for i,state in enumerate(shells[r]):
                for j in range(D):
                    for di in (-1,1):
                        ns=list(state); ns[j] += di
                        nradius=max(abs(a) for a in ns)
                        pa=pv if nradius>r else mp.mpf(1)
                        if nradius>r:
                            B[i,i] -= weight*(1-pa)
                        if nradius>=N:
                            continue
                        orbit=tuple(sorted(abs(z) for z in ns))
                        t=nradius; coeff=-weight*pa
                        if t==r: B[i,where[r][orbit]] += coeff
                        elif t==r+1: U[i,where[r+1][orbit]] += coeff
                        elif t==r-1: L[i,where[r-1][orbit]] += coeff
                        else: raise RuntimeError('Unexpected radial jump')
            rhs=mp.matrix([1]*nr)
            if r:
                B = B + L*beta[r-1]
                rhs = rhs - L*alpha[r-1]
            alpha[r]=mp.lu_solve(B,rhs)
            if U is not None:
                beta[r]=mp.matrix(nr,nxt)
                for j in range(nxt):
                    solution=mp.lu_solve(B, mp.matrix([-U[i,j] for i in range(nr)]))
                    for i in range(nr): beta[r][i,j]=solution[i]
        vector=alpha[-1]
        for r in range(N-2,-1,-1):
            vector=alpha[r]+beta[r]*vector
        return float(vector[0])


### 5. Validation
The 25 base checks cover analytic identities, agreement with a full-lattice solve,
a high-precision numerical reference and finite-depth dimensional reversal.
Four selected cases are checked at both 60 and 90 digits: $(D,N,p)=(2,23,0.2)$,
$(3,8,0.2)$, $(2,25,0.15)$ and $(3,10,0.15)$.
The paired results must agree to relative tolerance $10^{-15}$ and the float64 result to $10^{-10}$.

The reference function returns a float64 value, so paired-precision comparisons test agreement
at double precision, not agreement to 60 digits. Residual and LU-ordering checks do not bound forward error.

Two additional tests check the $N=2$ reversal threshold $p=(\sqrt5-1)/2$ and
the $p=1$ Brownian constant $c_2\approx1.1787$.
The latter is an additional true-chain check: $H_2(N)/N^2$ is compared with $c_2$,
whereas the quotient satisfies $Q_D(N)=N^2$ at $p=1$ (Remark 2).


In [ ]:
def full_state_direct_hitting_time(N, p, D):
    """Independent full-lattice solve for small-N validation.

    Uses no symmetry lumping or excursion decomposition. The direct system
    becomes poorly conditioned at large N or small p; use for small-N tests.
    """
    states = list(itertools.product(range(-N+1, N), repeat=D))
    ids = {x:i for i,x in enumerate(states)}
    rr, cc, vv = [], [], []
    for i,x in enumerate(states):
        rr.append(i); cc.append(i); vv.append(1.)
        rad = max(abs(z) for z in x)
        for j in range(D):
            for direction in (-1,1):
                y = list(x); y[j] += direction
                yr = max(abs(z) for z in y)
                accept = p if yr > rad else 1.
                if accept < 1:
                    rr.append(i); cc.append(i); vv.append(-(1-accept)/(2*D))
                if yr < N:
                    rr.append(i); cc.append(ids[tuple(y)])
                    vv.append(-accept/(2*D))
    M = coo_matrix((vv,(rr,cc)),shape=(len(states),len(states))).tocsc()
    T = spsolve(M, np.ones(len(states)))
    return float(T[ids[(0,)*D]])


def run_tests():
    checks=[]
    def test(name, value, reference, rtol=1e-9):
        relative = abs(value-reference)/max(1.0,abs(reference))
        if not np.isfinite(relative) or relative > rtol:
            raise AssertionError(f'{name}: {value}, expected {reference}, rel={relative}')
        checks.append({'test':name, 'actual':value,
                       'reference':reference, 'relative_difference':relative,
                       'tolerance':rtol, 'passed':True})

    # Analytic special cases.
    for p in (0.2, 0.55, 0.9):
        for N in (2,5,15,25):
            test(f'D=1 matches Q, N={N},p={p}',
                 true_hitting_time(N,p,1,False)['H_true'],
                 quotient_hitting_time(N,p,1))
    for p in (0.2,0.9):
        test(f'N=2,D=1 exact rational,p={p}',
             true_hitting_time(2,p,1)['H_true'], (1+3*p)/p**2)
        test(f'N=2,D=2 exact rational,p={p}',
             true_hitting_time(2,p,2)['H_true'],
             (5*p*p+12*p+1)/(p*p*(p+3)))
    for D in (1,2,3):
        test(f'quotient p=1 equals N^2, D={D}',
             quotient_hitting_time(7,1,D), 49)

    # Check exact orbit lumping against a truly independent full-state solve.
    for D, N, p in ((1,4,.55),(2,3,.2),(2,4,.9),(3,3,.55)):
        test(f'full-lattice vs orbit D={D},N={N},p={p}',
             true_hitting_time(N,p,D)['H_true'],
             full_state_direct_hitting_time(N,p,D))

    # Numerical reference from a 60-digit block-Thomas solve.
    test('High-precision benchmark D=2,N=23,p=.2',
         true_hitting_time(23,.2,2)['H_true'],
         1.1905397816993633e15,rtol=3e-12)

    # A true-chain dimensional reversal at finite N, not the quotient.
    H1=true_hitting_time(2,.9,1)['H_true']
    H2=true_hitting_time(2,.9,2)['H_true']
    if H2 <= H1:
        raise AssertionError('Expected finite-N counterexample not reproduced')
    checks.append({'test':'finite-N crossover D2 > D1 at N=2,p=.9',
                   'actual':H2, 'reference':H1,
                   'relative_difference':H2/H1-1,
                   'tolerance':None, 'passed':True})
    print(f'PASS: {len(checks)} analytic, full-lattice, and numerical-reference checks.')
    print(f"Hard benchmark: {true_hitting_time(23,.2,2)['H_true']:.16e}")
    return checks




unit_checks = run_tests()
def paired_precision_audits():
    cases = list(AUDIT_CASES) + ([(3, 25, 0.15)] if RUN_HARDEST_D3_AUDIT else [])
    rows = []
    for D, N, p in cases:
        tic = time.perf_counter()
        ref = {d: high_precision_block_reference(N, p, D, d) for d in AUDIT_DPS}
        lo, hi = ref[min(AUDIT_DPS)], ref[max(AUDIT_DPS)]
        stable = true_hitting_time(N, p, D)['H_true']
        precision_change = abs(lo/hi - 1)
        delta = abs(stable/hi - 1)
        ok = precision_change < 1e-15 and delta < 1e-10
        rows.append({'D': D, 'N': N, 'p': p, 'dps_low': min(AUDIT_DPS), 'dps_high': max(AUDIT_DPS),
                     'H_mpmath_low': lo, 'H_mpmath_high': hi, 'H_regenerative': stable,
                     'precision_change': precision_change, 'relative_difference': delta,
                     'runtime_seconds': time.perf_counter() - tic, 'passed': ok})
        if not ok:
            raise AssertionError(f'Paired audit failed D={D},N={N},p={p}: '
                                 f'precision change {precision_change:.1e}, float64 diff {delta:.1e}')
        print(f'PASS {min(AUDIT_DPS)}/{max(AUDIT_DPS)}-digit audit D={D} N={N:2d} p={p}: '
              f'precision change {precision_change:.1e}, float64 vs mpmath {delta:.1e} '
              f'({rows[-1]["runtime_seconds"]:.0f}s)')
    return rows

mp_checks = paired_precision_audits() if RUN_MPMATH_AUDIT else []

def additional_tests():
    out = []
    def rec(name, ok, detail):
        if not ok: raise AssertionError(f'{name}: {detail}')
        out.append(dict(test=name, passed=True, detail=detail))
    H = lambda N, p, D: true_hitting_time(N, p, D)['H_true']
    g = (5**0.5 - 1)/2
    lo, hi = H(2, g-1e-3, 2)/H(2, g-1e-3, 1), H(2, g+1e-3, 2)/H(2, g+1e-3, 1)
    rec('N=2 reversal at p=(sqrt5-1)/2', lo < 1 < hi, f'{lo:.6f} -> {hi:.6f}')
    c2 = H(100, 1.0, 2)/100**2
    rec('p=1 Brownian constant c_2', abs(c2 - 1.1787) < 2e-3, f'H/N^2={c2:.5f}')
    print(f'PASS: {len(out)} additional true-chain tests.')
    return out

extra_checks = additional_tests()


### 6. Compute the shared dataset
Compute the union of the configured figure grids with `true_hitting_time` and `quotient_hitting_time`.
The `used_in` column identifies supplementary panels S1, S2a, S2b and S3.
Figures 4 and 5 read subsets of these same data. Solver acceptance failures raise exceptions.


In [ ]:
used = {}
def want(D, N, p, tag): used.setdefault((D, N, round(float(p), 10)), set()).add(tag)
for D in SUPP1_D:
    for p in SUPP1_P:
        for N in SUPP1_N: want(D, N, p, 'supp_s1')
for D in SUPP2_D:
    for p in SUPP2A_P: want(D, SUPP2A_N, p, 'supp_s2a')
    for N in SUPP2B_N: want(D, N, SUPP2B_P, 'supp_s2b')
for D in SUPP3_D:
    for N in SUPP3_N:
        for p in SUPP3_P: want(D, N, p, 'supp_s3')
requested = set(used)

records=[]
start = time.perf_counter()
for D,N,p in sorted(requested):
    audit=true_hitting_time(N,p,D,check_alt=CHECK_ALTERNATE_LU_ORDERING)
    Q=quotient_hitting_time(N,p,D)
    H=audit['H_true']
    records.append({
        'D':D,'N':N,'p':p,
        'Q_quotient':Q,'H_true':H,
        'signed_error_pct':100*(Q/H-1),
        'H_over_Q':H/Q,
        'normalized_Q':Q*(p**N)*N**(D-1),
        'normalized_H':H*(p**N)*N**(D-1),
        **audit,
        'method': 'exact symmetry-orbit + regenerative sparse LU',
        'precision': 'float64',
        'used_in': ';'.join(sorted(used[(D,N,p)])),
    })
results=pd.DataFrame.from_records(records).sort_values(['D','p','N']).reset_index(drop=True)
elapsed=time.perf_counter()-start
assert len(results)==len(requested)
assert results[['D','N','p']].duplicated().sum()==0
assert np.isfinite(results[['Q_quotient','H_true']]).to_numpy().all()
assert (results['H_true']>0).all()
assert (results['Q_quotient']>0).all()
assert results.method.nunique()==1 and results.precision.eq('float64').all()
if (results['signed_error_pct']>1e-8).any():
    print('Computed points with Q > H (relative tolerance 1e-10):',
          int((results['signed_error_pct'] > 1e-8).sum()))
print(f'Computed {len(results)} validated parameter combinations in {elapsed:.2f} seconds.')
print('H_D: symmetry-orbit + regenerative sparse LU, float64. Q_D: closed-form quotient sum.')
print('Largest relative difference between independent sparse LU orderings:',
      f"{results['alternative_relative_difference'].max():.3e}")
print('Largest componentwise backward error:', f"{results[['error_h','error_g']].max().max():.3e}")
results.to_csv(OUTPUT_DIR/'data'/'all_results.csv', index=False, float_format='%.16g')
pd.DataFrame(unit_checks + extra_checks).to_csv(OUTPUT_DIR/'data'/'unit_checks.csv', index=False)
pd.DataFrame(mp_checks).to_csv(OUTPUT_DIR/'data'/'high_precision_crosschecks.csv', index=False)


### 7. Supplementary figures
Additional repository plots S1–S3, separate from the paper's Figures 4 and 5.
Each is read from `all_results.csv` and saved as PDF, SVG and PNG in `figures/supplementary/`.


In [ ]:
R = pd.read_csv(OUTPUT_DIR/'data'/'all_results.csv')
def sel(tag, **kw):
    d = R[R.used_in.str.contains(tag)]
    for k, v in kw.items(): d = d[np.isclose(d[k], v)]
    return d
def save(fig, stem):
    for ext in ('pdf', 'svg', 'png'):
        fig.savefig(OUTPUT_DIR/'figures'/'supplementary'/f'{stem}.{ext}', dpi=250, facecolor='white')
    plt.show(); plt.close(fig); print('Saved', stem)

P_COL = {0.20: '#17679a', 0.55: '#c56b20', 0.90: '#16816d'}
D_COL = {1: '#2b5aa8', 2: '#8c2d2d', 3: '#2e7d32'}; D_MK = {1: 'o', 2: 's', 3: '^'}
N_COL = {5: '#9ecae1', 15: '#4292c6', 25: '#08306b'}; N_MK = {5: 'o', 15: 's', 25: '^'}
ERR_YLIM = (min(-5, 1.08*R[R.D > 1].signed_error_pct.min()), 2)   # shared by Supplementary Figures S1 and S3


#### Supplementary Figure S1 — escape time and error versus basin depth


In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(7.5, 5.0), sharex=True,
                       gridspec_kw=dict(height_ratios=[1.3, 1], hspace=0.36, wspace=0.55))
for c, D in enumerate(SUPP1_D):
    for p in SUPP1_P:
        a = sel('supp_s1', D=D, p=p).sort_values('N')
        ax[0, c].semilogy(a.N, a.H_true, '-', color=P_COL[p], lw=1.7)
        if D > 1: ax[0, c].semilogy(a.N, a.Q_quotient, '--', color=P_COL[p], lw=1.2)
        ax[1, c].plot(a.N, a.signed_error_pct, 'o-', color=P_COL[p], ms=2.6, lw=1.4)
    ax[0, c].set_title(f'$D={D}$'); ax[1, c].axhline(0, color='k', lw=.7, alpha=.45)
    ax[1, c].set_xlabel('Basin depth $N$'); ax[1, c].set_ylim(*ERR_YLIM)
    for r in (0, 1): ax[r, c].grid(True, alpha=.2)
ax[0, 0].set_ylabel('Mean escape time\n(attempted steps)')
ax[1, 0].set_ylabel(r'$(Q_D-H_D)/H_D$ (%)')
ax[1, 0].text(.5, .45, '$Q_1=H_1$ exactly', transform=ax[1, 0].transAxes, ha='center', fontsize=7.5, color='.3')
for x in ax.flat:
    x.set_xticks(range(5, max(SUPP1_N) + 1, 5))
    x.tick_params(labelbottom=True, labelleft=True)
    x.set_xlabel('Basin depth $N$')
for x in ax[0]: x.set_ylabel('Mean escape time\n(attempted steps)')
for x in ax[1]: x.set_ylabel(r'$(Q_D-H_D)/H_D$ (%)')
h = [Line2D([0], [0], color=P_COL[p], lw=2, label=f'$p={p:g}$') for p in SUPP1_P]
h += [Line2D([0], [0], color='k', lw=1.7, label='True $H_D$'),
      Line2D([0], [0], color='k', lw=1.2, ls='--', label='Quotient $Q_D$')]
fig.legend(handles=h, loc='lower center', ncol=5, frameon=False, bbox_to_anchor=(.5, -.02))
fig.subplots_adjust(bottom=.15)
save(fig, 'fig_s1_escape_and_error_vs_depth')


#### Supplementary Figure S2 — dimensional dependence: true walk (top) and quotient (bottom)


In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(7.3, 5.4), sharex='col', sharey='col',
                       gridspec_kw=dict(hspace=.36, wspace=.42))
for D in SUPP2_D:
    a = sel('supp_s2a', D=D).sort_values('p'); b = sel('supp_s2b', D=D).sort_values('N')
    for r, col in ((0, 'H_true'), (1, 'Q_quotient')):
        ax[r, 0].semilogy(a.p, a[col], '-', marker=D_MK[D], ms=2.4, lw=1.5, color=D_COL[D], label=f'$D={D}$')
        ax[r, 1].semilogy(b.N, b[col], '-', marker=D_MK[D], ms=2.8, lw=1.5, color=D_COL[D], label=f'$D={D}$')

def ratio(tag, col, key, val):
    d = sel(tag); d = d[np.isclose(d[key], val)].set_index('D')[col]; return d[1]/d[3]
pa, nb = min(SUPP2A_P), max(SUPP2B_N)
for r, col, lab in ((0, 'H_true', 'true walk'), (1, 'Q_quotient', 'formula')):
    ax[r, 0].text(.97, .95, f'$p={pa:g}$: 3D {ratio("supp_s2a", col, "p", pa):.0f}× faster than 1D',
                  transform=ax[r, 0].transAxes, ha='right', va='top', fontsize=7.3, color='.25')
    ax[r, 1].text(.97, .05, f'$N={nb}$: 3D {ratio("supp_s2b", col, "N", nb):.0f}× faster than 1D',
                  transform=ax[r, 1].transAxes, ha='right', va='bottom', fontsize=7.3, color='.25')
ax[0, 0].set_title(f'$N={SUPP2A_N}$, vary $p$'); ax[0, 1].set_title(f'$p={SUPP2B_P:g}$, vary $N$')
ax[0, 0].set_ylabel('True walk $H_D$\n(attempted steps)')
ax[1, 0].set_ylabel('Quotient $Q_D$\n(attempted steps)')
ax[1, 0].set_xlabel('Uphill acceptance probability $p$'); ax[1, 1].set_xlabel('Basin depth $N$')
for x in ax.flat:
    x.grid(True, alpha=.2); x.tick_params(labelbottom=True, labelleft=True)
for x in ax[:, 1]: x.set_xticks(range(5, max(SUPP2B_N) + 1, 5))
for r in (0, 1):
    ax[r, 0].set_xlabel('Uphill acceptance probability $p$'); ax[r, 1].set_xlabel('Basin depth $N$')
ax[0, 1].set_ylabel('True walk $H_D$\n(attempted steps)')
ax[1, 1].set_ylabel('Quotient $Q_D$\n(attempted steps)')
ax[0, 0].legend(frameon=False, loc='lower left'); ax[0, 1].legend(frameon=False, loc='upper left')
save(fig, 'fig_s2_dimensional_speedup')


#### Supplementary Figure S3 — escape time and error versus acceptance probability


In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(7.5, 5.0), sharex=True,
                       gridspec_kw=dict(height_ratios=[1.3, 1], hspace=.36, wspace=.55))
for c, D in enumerate(SUPP3_D):
    for N in SUPP3_N:
        a = sel('supp_s3', D=D, N=N).sort_values('p')
        ax[0, c].semilogy(a.p, a.H_true, '-', color=N_COL[N], lw=1.7)
        if D > 1: ax[0, c].semilogy(a.p, a.Q_quotient, '--', color=N_COL[N], lw=1.2)
        ax[1, c].plot(a.p, a.signed_error_pct, '-', marker=N_MK[N], color=N_COL[N], ms=2.6, lw=1.4)
    ax[0, c].set_title(f'$D={D}$'); ax[1, c].axhline(0, color='k', lw=.7, alpha=.45)
    ax[1, c].set_ylim(*ERR_YLIM)
    for r in (0, 1): ax[r, c].grid(True, alpha=.2)
ax[0, 0].set_ylabel('Mean escape time\n(attempted steps)')
ax[1, 0].set_ylabel(r'$(Q_D-H_D)/H_D$ (%)')
ax[1, 0].text(.5, .45, '$Q_1=H_1$ exactly', transform=ax[1, 0].transAxes, ha='center', fontsize=7.5, color='.3')
for x in ax.flat:
    x.tick_params(labelbottom=True, labelleft=True); x.set_xlabel('Acceptance probability $p$')
for x in ax[0]: x.set_ylabel('Mean escape time\n(attempted steps)')
for x in ax[1]: x.set_ylabel(r'$(Q_D-H_D)/H_D$ (%)')
h = [Line2D([0], [0], color=N_COL[N], lw=2, marker=N_MK[N], ms=4, label=f'$N={N}$') for N in SUPP3_N]
h += [Line2D([0], [0], color='k', lw=1.7, label='True $H_D$'),
      Line2D([0], [0], color='k', lw=1.2, ls='--', label='Quotient $Q_D$')]
fig.legend(handles=h, loc='lower center', ncol=5, frameon=False, bbox_to_anchor=(.5, -.02))
fig.subplots_adjust(bottom=.15)
save(fig, 'fig_s3_escape_and_error_vs_p')


### 8. Paper Figures 4 and 5

These plots read `data/all_results.csv`. The default configuration contains all required points.
Changing the configuration to omit a paper-grid point raises an error with the missing parameters.

| Paper figure | PDF filename |
|---|---|
| Figure 4: true escape times and dimensional speedup | `fig_vi_dimensional_speedup.pdf` |
| Figure 5: true and quotient times, with signed discrepancy | `fig_vi_quotient_comparison.pdf` |

PDF, SVG and PNG versions are saved in `figures/`. The ZIP also includes the paper plot data,
captions and a manifest. Ten checks verify the numerical values quoted in Section VI at their displayed precision,
including the shallow-basin reversal example. Results are exported to `data/paper_value_checks.csv`.


In [ ]:
# Fixed paper grids, independent of supplementary figure settings.
PAPER_D = (1, 2, 3)
PAPER_P = (0.2, 0.55, 0.9)
PAPER_N = (2, 3, 5, 8, 11, 14, 17, 20, 23, 25)
PAPER_SPEEDUP_P = tuple(np.round(np.linspace(0.15, 0.9, 50), 6))
PAPER_SPEEDUP_N = tuple(range(5, 26))
paper_data = pd.read_csv(OUTPUT_DIR / 'data' / 'all_results.csv')
if paper_data[['D', 'N', 'p']].duplicated().any():
    raise ValueError('Duplicate parameter points in all_results.csv.')


def paper_row(D, N, p):
    a = paper_data[(paper_data.D == D) & (paper_data.N == N)
                   & np.isclose(paper_data.p, p, rtol=0, atol=1e-9)]
    if len(a) != 1:
        raise ValueError(f'Require one paper data point (D={D}, N={N}, p={p}); found {len(a)}. '
                         'Restore the default configuration and rerun data generation.')
    return a.iloc[0]


paper_depth_data = pd.DataFrame([paper_row(D, N, p)
    for D in PAPER_D for p in PAPER_P for N in PAPER_N])
paper_time_data = pd.DataFrame([paper_row(D, N, .55)
    for D in PAPER_D for N in PAPER_SPEEDUP_N])
paper_ratio_data = pd.DataFrame([paper_row(D, 20, p)
    for D in PAPER_D for p in PAPER_SPEEDUP_P])
for a in (paper_depth_data, paper_time_data, paper_ratio_data):
    if not np.isfinite(a[['H_true', 'Q_quotient']]).to_numpy().all() or (a[['H_true', 'Q_quotient']] <= 0).to_numpy().any():
        raise ValueError('Paper data must contain finite, positive escape times.')
np.testing.assert_allclose(paper_depth_data.signed_error_pct,
    100 * (paper_depth_data.Q_quotient / paper_depth_data.H_true - 1), rtol=1e-10, atol=1e-10)


def paper_save(fig, stem):
    for ext in ('pdf', 'svg', 'png'):
        fig.savefig(OUTPUT_DIR / 'figures' / f'{stem}.{ext}',
                    dpi=250, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)
    print('Paper figure:', OUTPUT_DIR / 'figures' / f'{stem}.pdf')


#### Figure 4 — dimensional speedup of the true walk

Left: $H_D(N)$ at $p=0.55$. Right: $H_1/H_2$ and $H_1/H_3$ at $N=20$; the dotted line marks no speedup. Both vertical axes are logarithmic. Supplementary Figure S2 also shows the shell-quotient means.


In [ ]:
with plt.rc_context(plt.rcParamsDefault):
    plt.rcParams.update({'font.size': 10, 'font.family': 'DejaVu Sans',
                         'mathtext.fontset': 'stix', 'pdf.fonttype': 42, 'svg.fonttype': 'none'})
    fig, ax = plt.subplots(1, 2, figsize=(7.3, 3.1), layout='constrained')
    colors = {1: '#2b5aa8', 2: '#8c2d2d', 3: '#2e7d32'}
    for D, marker in ((1, 'o'), (2, 's'), (3, '^')):
        a = paper_time_data[paper_time_data.D == D].sort_values('N')
        ax[0].semilogy(a.N, a.H_true, color=colors[D], marker=marker,
                       ms=3, lw=1.5, label=f'$D={D}$')
    ratios = paper_ratio_data.pivot(index='p', columns='D', values='H_true').sort_index()
    for D, marker in ((2, 's'), (3, '^')):
        ax[1].semilogy(ratios.index, ratios[1] / ratios[D], color=colors[D],
                       marker=marker, markevery=5, ms=3, lw=1.5,
                       label=rf'$H_1/H_{D}$')
    ax[0].set_title('(a) Escape time at $p=0.55$')
    ax[0].set_xlabel('Basin depth $N$')
    ax[0].set_ylabel('Mean escape time $H_D$\n(attempted steps)')
    ax[0].set_xticks([5, 10, 15, 20, 25])
    ax[1].set_title('(b) Dimensional speedup at $N=20$')
    ax[1].set_xlabel('Uphill acceptance probability $p$')
    ax[1].set_ylabel('Speedup $H_1/H_D$')
    ax[1].axhline(1, color='.5', ls=':', lw=1)
    for a in ax:
        a.grid(True, alpha=.2)
        a.legend(frameon=False, fontsize=9)
    paper_save(fig, 'fig_vi_dimensional_speedup')


#### Figure 5 — quotient comparison and signed relative discrepancy

Six-panel layout: $D=1,2,3$ across columns, escape times above, signed percentage discrepancy below. The one-dimensional panel records the exact equality $Q_1=H_1$.

In [ ]:
with plt.rc_context(plt.rcParamsDefault):
    plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 9, 'axes.titlesize': 10,
        'axes.labelsize': 9, 'legend.fontsize': 8, 'xtick.labelsize': 8,
        'ytick.labelsize': 8, 'pdf.fonttype': 42, 'ps.fonttype': 42,
        'figure.dpi': 130, 'savefig.bbox': 'tight'})
    P_COL = {0.20: '#17679a', 0.55: '#c56b20', 0.90: '#16816d'}
    ERR_YLIM = (min(-5, 1.08 * paper_data[paper_data.D > 1].signed_error_pct.min()), 2)
    fig, ax = plt.subplots(2, 3, figsize=(7.5, 5.0), sharex=True,
                           gridspec_kw=dict(height_ratios=[1.3, 1], hspace=0.36, wspace=0.55))
    for c, D in enumerate(PAPER_D):
        for p in PAPER_P:
            a = paper_depth_data[(paper_depth_data.D == D) & np.isclose(paper_depth_data.p, p)].sort_values('N')
            ax[0, c].semilogy(a.N, a.H_true, '-', color=P_COL[p], lw=1.7)
            if D > 1: ax[0, c].semilogy(a.N, a.Q_quotient, '--', color=P_COL[p], lw=1.2)
            ax[1, c].plot(a.N, a.signed_error_pct, 'o-', color=P_COL[p], ms=2.6, lw=1.4)
        ax[0, c].set_title(f'$D={D}$'); ax[1, c].axhline(0, color='k', lw=.7, alpha=.45)
        ax[1, c].set_xlabel('Basin depth $N$'); ax[1, c].set_ylim(*ERR_YLIM)
        for r in (0, 1): ax[r, c].grid(True, alpha=.2)
    ax[0, 0].set_ylabel('Mean escape time\n(attempted steps)')
    ax[1, 0].set_ylabel(r'$(Q_D-H_D)/H_D$ (%)')
    ax[1, 0].text(.5, .45, '$Q_1=H_1$ exactly', transform=ax[1, 0].transAxes, ha='center', fontsize=7.5, color='.3')
    for x in ax.flat:
        x.set_xticks(range(5, max(PAPER_N) + 1, 5))
        x.tick_params(labelbottom=True, labelleft=True)
        x.set_xlabel('Basin depth $N$')
    for x in ax[0]: x.set_ylabel('Mean escape time\n(attempted steps)')
    for x in ax[1]: x.set_ylabel(r'$(Q_D-H_D)/H_D$ (%)')
    h = [Line2D([0], [0], color=P_COL[p], lw=2, label=f'$p={p:g}$') for p in PAPER_P]
    h += [Line2D([0], [0], color='k', lw=1.7, label='True $H_D$'),
          Line2D([0], [0], color='k', lw=1.2, ls='--', label='Quotient $Q_D$')]
    fig.legend(handles=h, loc='lower center', ncol=5, frameon=False, bbox_to_anchor=(.5, -.02))
    fig.subplots_adjust(bottom=.15)
    paper_save(fig, 'fig_vi_quotient_comparison')

In [ ]:
# Record the paper's plotted data and verify its quoted rounded values.
paper_time_data.to_csv(OUTPUT_DIR / 'data' / 'paper_escape_vs_depth.csv', index=False)
paper_depth_data.to_csv(OUTPUT_DIR / 'data' / 'paper_quotient_comparison.csv', index=False)
paper_speedups = ratios.rename(columns={1: 'H1', 2: 'H2', 3: 'H3'}).copy()
paper_speedups['H1_over_H2'] = paper_speedups.H1 / paper_speedups.H2
paper_speedups['H1_over_H3'] = paper_speedups.H1 / paper_speedups.H3
paper_speedups.to_csv(OUTPUT_DIR / 'data' / 'paper_dimensional_speedup.csv')
paper_checks = [
    ('H1/H2 at N=25, p=.55', paper_row(1,25,.55).H_true / paper_row(2,25,.55).H_true, 6.6, 1),
    ('H1/H3 at N=25, p=.55', paper_row(1,25,.55).H_true / paper_row(3,25,.55).H_true, 28.4, 1),
    ('H1/H3 at N=20, p=.15', paper_row(1,20,.15).H_true / paper_row(3,20,.15).H_true, 243, 0),
    ('H1/H3 at N=20, p=.9', paper_row(1,20,.9).H_true / paper_row(3,20,.9).H_true, 1.14, 2),
    ('H1 at N=2, p=.9', paper_row(1,2,.9).H_true, 4.57, 2),
    ('H2 at N=2, p=.9', paper_row(2,2,.9).H_true, 5.02, 2),
    ('delta D=2, N=25, p=.2', paper_row(2,25,.2).signed_error_pct, -1.4, 1),
    ('delta D=3, N=25, p=.2', paper_row(3,25,.2).signed_error_pct, -2.8, 1),
    ('delta D=2, N=25, p=.9', paper_row(2,25,.9).signed_error_pct, -14.5, 1),
    ('delta D=3, N=25, p=.9', paper_row(3,25,.9).signed_error_pct, -25.5, 1),
]
paper_check_rows = []
for name, value, quoted, decimals in paper_checks:
    matched = round(value, decimals) == quoted
    if not matched:
        raise AssertionError((name, value, quoted))
    paper_check_rows.append(dict(quantity=name, computed=float(value),
                                 paper_value=quoted, decimal_places=decimals, passed=matched))
pd.DataFrame(paper_check_rows).to_csv(OUTPUT_DIR / 'data' / 'paper_value_checks.csv', index=False)
print('All Section VI quoted values agree at the displayed precision.')
paper_info = {
    'data_source': 'data/all_results.csv',
    'paper_figures': ['figures/fig_vi_dimensional_speedup.pdf', 'figures/fig_vi_quotient_comparison.pdf'],
    'generated_formats': ['pdf', 'svg', 'png'],
    'speedup_panel_a': {'p': .55, 'N': list(PAPER_SPEEDUP_N), 'D': list(PAPER_D)},
    'speedup_panel_b': {'N': 20, 'p': list(PAPER_SPEEDUP_P), 'ratios': ['H1/H2', 'H1/H3']},
    'quotient_comparison': {'N': list(PAPER_N), 'p': list(PAPER_P), 'D': list(PAPER_D)},
    'quoted_value_checks_passed': len(paper_checks),
}
(OUTPUT_DIR / 'paper_figures_manifest.json').write_text(json.dumps(paper_info, indent=2) + '\n')
(OUTPUT_DIR / 'paper_figure_captions.md').write_text(
    'Figure 4. Dimensional dependence of the true Metropolis escape time. '
    '(a) Mean escape time versus depth at p=0.55, D=1,2,3, N=5,...,25. '
    '(b) Speedup H1/HD at N=20 for 50 acceptance probabilities from 0.15 to 0.9. '
    'Both vertical axes are logarithmic.\n\n'
    'Figure 5. True H_D (solid) and quotient Q_D (dashed) versus depth (top), '
    'with signed relative discrepancy 100(Q_D/H_D-1) in percent (bottom). '
    'Columns: D=1,2,3. Colors: p=0.2,0.55,0.9. Times count attempted moves. '
    'In one dimension the means coincide exactly.\n')


### 9. Manifest and download


In [ ]:
manifest = dict(
    created_utc=datetime.now(timezone.utc).isoformat(),
    value_method='exact symmetry-orbit + regenerative sparse LU, float64, for all plotted H_D values',
    quotient_method='Theorem 1, Eq. (17), closed-form shell-quotient sum',
    dataset='data/all_results.csv (single source for all figures)',
    model='V(x)=||x||_inf on Z^D, nearest-neighbour proposals, uphill accept p, absorb on shell N',
    time_units='attempted moves including rejected proposals',
    unique_points=len(R),
    tests_passed=len(unit_checks) + len(extra_checks),
    paired_precision_audits_compare_only=[(r['D'], r['N'], r['p'], r['dps_low'], r['dps_high']) for r in mp_checks],
    paper_quoted_value_checks_passed=len(paper_checks),
    supplementary_figures=['figures/supplementary/fig_s1_escape_and_error_vs_depth.pdf',
                           'figures/supplementary/fig_s2_dimensional_speedup.pdf',
                           'figures/supplementary/fig_s3_escape_and_error_vs_p.pdf'],
    observed_quotient_comparison=dict(
        relative_tolerance=1e-10,
        points_Q_greater_than_H=int((R.signed_error_pct > 1e-8).sum()),
        max_signed_error_pct=float(R.signed_error_pct.max())),
    config=dict(SUPP1_N=SUPP1_N, SUPP1_P=SUPP1_P, SUPP2A_N=SUPP2A_N, SUPP2A_P_range=[min(SUPP2A_P), max(SUPP2A_P), len(SUPP2A_P)],
                SUPP2B_P=SUPP2B_P, SUPP2B_N=SUPP2B_N, SUPP3_N=SUPP3_N, SUPP3_P=SUPP3_P),
    versions=dict(python=sys.version.split()[0], numpy=np.__version__, scipy=scipy.__version__,
                  pandas=pd.__version__, matplotlib=matplotlib.__version__, mpmath=mp.__version__),
    numerical_notes=[
        'Comparisons of Q_D and H_D describe the configured grid, not a general inequality.',
        'Residual and LU-ordering checks do not bound forward error.',
        'High-precision references are rounded to float64 before comparison.'])
(OUTPUT_DIR/'manifest.json').write_text(json.dumps(manifest, indent=2, default=str) + '\n')
zp = OUTPUT_DIR.parent/'metropolis_results.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUTPUT_DIR.rglob('*')):
        if f.is_file(): z.write(f, f.relative_to(OUTPUT_DIR.parent))
print('ZIP:', zp)
try:
    from google.colab import files; files.download(str(zp))
except ImportError:
    print('Not in Colab: download the ZIP from the path above.')
